In [ ]:
%pip install pypdf

In [ ]:
from pypdf import PdfReader
pdf_path = "/Workspace/Users/yaminibhole20@gmail.com/Databricks_Genie_Rag_assistant/Data_Ingestion/policy.pdf"
reader = PdfReader(pdf_path)

text = " "
for page in reader.pages:
    text += page.extract_text()
print(text[:1000])

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Enhanced chunking strategy with sentence awareness and overlap
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # Maximum chunk size in characters
    chunk_overlap=50,  # Overlap between chunks for context continuity
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]  # Try to split on paragraph, then newline, then sentence, then word
)

chunks = text_splitter.split_text(text)

print(f"Total chunks created: {len(chunks)}")
print(f"\nFirst chunk (with sentence boundaries respected):")
print(f"{chunks[0]}")
print(f"\nSecond chunk (notice the overlap):")
print(f"{chunks[1]}")

In [ ]:
chunk_data = [(i, chunk) for i, chunk in enumerate(chunks)]
chunk_df = spark.createDataFrame(
    chunk_data,
    ["id", "content"]
)
chunk_df.write.mode("overwrite").saveAsTable("genie.demo.document_chunks")
display(chunk_df)